# CenpAI synthetic training data (Phase 5, step 7)

Makes four kinds of synthetic images, each into **its own training-only folder** on Drive, so `notebooks/train_detector_compare.ipynb` can switch each kind on or off per run (`SYNTHETIC` setting):

| Folder | What it is | Question it answers (matrix) |
|---|---|---|
| `variants` | Offline variants of training images with plates: dark, backlit, motion blur, perspective, noise and compression (Albumentations, YOLO boxes, `min_visibility=0.3`) | S1: night and backlight |
| `copypaste` | Training plates cut out and pasted onto the no-plate training images (keyboards and others), including half off the edge | S2: edge plates, false masks |
| `screen` | The screen effect: moire, glare and a colour shift, and training photos shown inside a screen frame | S3: plates on screens |
| `rendered` | Philippine-style plates drawn with made-up text and warped onto backgrounds | S4: more PH layouts |

Rules (`docs/protocols/data_and_phone_test.md`):
- **Reads only the training images** of dataset v2 (`images/train`). Validation and test images are never opened. The family-car test videos are never used, and nothing is extracted from them.
- Synthetic images go to training only. Run `ml.dataset.leak_report(v2, extra_train=[...])` before training on them (the last cell here does it, and the training notebook does it again).
- Plate text on rendered plates is random and made up. No keys or tokens; every number printed is measured in the run.
- A folder that already exists is never overwritten. Change `SUFFIX` to make another version.

Reuses the ideas of the owner's `tools/synth_plates.py` (rendered plate layout, scene backdrops, plate warp, camera effects); that file is not part of the repo and is not imported.

**Before running** (Colab, CPU is enough): dataset v2 must be at `MyDrive/CenpAI/data/processed/dataset-v2.zip`.

In [ ]:
# 1. Setup
!pip -q install albumentations opencv-python-headless pillow pyyaml
import os, sys, json, csv, shutil, zipfile, subprocess, random, inspect, math
from pathlib import Path
import numpy as np, cv2
import albumentations as A
from PIL import Image, ImageDraw, ImageFont
print("albumentations", A.__version__, "| opencv", cv2.__version__)

from google.colab import drive
drive.mount("/content/drive")

REPO_BRANCH = "develop"  #@param {type:"string"}
REPO = Path("/content/cenpai")
if not REPO.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, "https://github.com/RedGla/cenpai", str(REPO)], check=True)
sys.path.insert(0, str(REPO))
from ml import dataset as ds

ROOT = Path("/content/drive/MyDrive/CenpAI")
V2_ZIP = ROOT / "data" / "processed" / "dataset-v2.zip"
SYN_DIR = ROOT / "data" / "synthetic"
SYN_DIR.mkdir(parents=True, exist_ok=True)
WORK = Path("/content/work")
V2 = WORK / "dataset-v2"

## 2. Settings

Counts are per kind. The protocol keeps synthetic data to at most about a third to a half of the training images, and rendered plates to a small share; the last cell prints the shares. `SEED` makes a run repeatable.

In [ ]:
SEED = 0                    #@param {type:"integer"}
SUFFIX = ""                 #@param {type:"string"}
# SUFFIX is added to every folder name (variants-b, ...), so a new version never overwrites an old one.
N_VARIANTS = 1500           #@param {type:"integer"}
N_COPYPASTE = 1500          #@param {type:"integer"}
N_SCREEN = 1000             #@param {type:"integer"}
N_RENDERED = 600            #@param {type:"integer"}
MAX_SIDE = 640              #@param {type:"integer"}
# MAX_SIDE: the long side of every synthetic image (the training size), so files stay small.
RUN_KINDS = ["variants", "copypaste", "screen", "rendered"]   # remove a name to skip that kind

## 3. Training images only

Unzips dataset v2 to local disk and reads **only** `images/train` and `labels/train`. It splits them into images with at least one plate box and images with none (keyboards and other no-plate images). The no-plate images are the backgrounds for copy-paste and rendered plates. Some no-plate images from sources with unlabeled plates could still show a plate; the cell that previews samples is the place to catch that.

In [ ]:
if not (V2 / "manifest.csv").is_file():
    assert V2_ZIP.is_file(), f"{V2_ZIP} not found: run notebooks/build_dataset_v2.ipynb first"
    shutil.copy(V2_ZIP, "/content/v2.zip")
    with zipfile.ZipFile("/content/v2.zip") as z:
        z.extractall(V2)
    os.remove("/content/v2.zip")
TRAIN_IMG = V2 / "images" / "train"
TRAIN_LBL = V2 / "labels" / "train"

def read_boxes(lbl):
    # YOLO rows "0 cx cy w h" -> list of (cx, cy, w, h)
    if not lbl.is_file():
        return []
    out = []
    for line in lbl.read_text().splitlines():
        p = line.split()
        if len(p) == 5:
            out.append(tuple(float(v) for v in p[1:]))
    return out

train_files = sorted(p for p in TRAIN_IMG.iterdir() if p.suffix.lower() in ds.IMAGE_EXTS)
with_plates, no_plates = [], []
for p in train_files:
    (with_plates if read_boxes(TRAIN_LBL / (p.stem + ".txt")) else no_plates).append(p)
print(f"training images: {len(train_files)} | with a plate: {len(with_plates)} | no plate: {len(no_plates)}")
assert with_plates, "no training images with plates"
from collections import Counter
print("no-plate images by source:", dict(Counter(p.name.split('__')[0] for p in no_plates)))

def load(path, max_side=MAX_SIDE):
    im = cv2.imdecode(np.fromfile(str(path), np.uint8), cv2.IMREAD_COLOR)
    im = cv2.cvtColor(im, cv2.COLOR_BGR2RGB)
    h, w = im.shape[:2]
    s = max_side / max(h, w)
    if s < 1:
        im = cv2.resize(im, (round(w * s), round(h * s)), interpolation=cv2.INTER_AREA)
    return im

def clip_box(cx, cy, w, h, eps=1e-6):
    # normalised YOLO box -> clipped to the image, or None if empty
    x0, y0, x1, y1 = max(0.0, cx - w / 2), max(0.0, cy - h / 2), min(1.0, cx + w / 2), min(1.0, cy + h / 2)
    if x1 - x0 < eps or y1 - y0 < eps:
        return None
    return ((x0 + x1) / 2, (y0 + y1) / 2, x1 - x0, y1 - y0)

class KindWriter:
    # One output folder per kind: images/, labels/, parents.csv (synthetic file -> training image it came from), manifest.json
    def __init__(self, kind):
        self.name = kind + SUFFIX
        self.dir = SYN_DIR / self.name
        assert not self.dir.exists(), f"{self.dir} already exists: set SUFFIX to make another version (folders are never overwritten)"
        (self.dir / "images").mkdir(parents=True)
        (self.dir / "labels").mkdir(parents=True)
        self.kind, self.n, self.boxes, self.neg, self.rows = kind, 0, 0, 0, []
        self.rng = random.Random(f"{SEED}:{kind}")

    def add(self, img_rgb, boxes_px_or_norm, parents, norm=True, quality=None):
        h, w = img_rgb.shape[:2]
        rows = []
        for b in boxes_px_or_norm:
            if not norm:
                x0, y0, x1, y1 = b
                b = ((x0 + x1) / 2 / w, (y0 + y1) / 2 / h, (x1 - x0) / w, (y1 - y0) / h)
            b = clip_box(*b)
            if b:
                rows.append("0 %.6f %.6f %.6f %.6f" % b)
        name = f"syn-{self.kind}__{self.n:05d}"
        ok, buf = cv2.imencode(".jpg", cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR),
                               [cv2.IMWRITE_JPEG_QUALITY, quality or self.rng.randint(75, 95)])
        assert ok
        (self.dir / "images" / f"{name}.jpg").write_bytes(buf.tobytes())
        (self.dir / "labels" / f"{name}.txt").write_text("\n".join(rows) + ("\n" if rows else ""))
        self.rows.append((f"{name}.jpg", ";".join(parents)))
        self.n += 1; self.boxes += len(rows); self.neg += (not rows)
        return len(rows)

    def close(self, extra=None):
        with open(self.dir / "parents.csv", "w", newline="") as f:
            w = csv.writer(f); w.writerow(["file", "training_images"]); w.writerows(self.rows)
        m = dict(kind=self.kind, folder=self.name, images=self.n, boxes=self.boxes, no_plate_images=self.neg, seed=SEED,
                 max_side=MAX_SIDE, source="dataset-v2 images/train only", albumentations=A.__version__, **(extra or {}))
        (self.dir / "manifest.json").write_text(json.dumps(m, indent=2) + "\n")
        print(json.dumps(m, indent=2))
        return m

MANIFESTS = {}

## 4. `variants`: offline variants (Albumentations)

Each output starts from a training image with a plate and applies one condition the phone failed on: **dark**, **backlit** (glare plus darker subject), **motion blur**, **perspective**, or **noise and compression**. Boxes go through the transform in YOLO format with `min_visibility=0.3`, so a plate that is mostly cut off by the transform loses its box, and an image that loses every box is skipped. The helper builds each transform from the arguments the installed Albumentations version accepts (they were renamed between 1.x and 2.x).

In [ ]:
def make(cls, *alternatives, **fixed):
    # Albumentations renamed arguments between versions: use the first set the installed class really has.
    params = set(inspect.signature(cls.__init__).parameters)
    for kw in alternatives:
        if set(kw) <= params:
            return cls(**fixed, **kw)
    raise TypeError(f"{cls.__name__}: none of {alternatives} match {sorted(params)}")

def glare(img, rng, strength=170):
    h, w = img.shape[:2]
    yy, xx = np.mgrid[0:h, 0:w]
    gx, gy = rng.uniform(0, w), rng.uniform(0, h)
    blob = np.exp(-((xx - gx) ** 2 + (yy - gy) ** 2) / (2 * (rng.uniform(0.15, 0.35) * max(w, h)) ** 2))
    return np.clip(img.astype(np.float32) + strength * blob[..., None], 0, 255).astype(np.uint8)

def variant_transform(kind):
    bbox = A.BboxParams(format="yolo", label_fields=["labels"], min_visibility=0.3)
    if kind == "dark":
        ts = [make(A.RandomBrightnessContrast, dict(brightness_limit=(-0.65, -0.35), contrast_limit=(-0.2, 0.2)), p=1.0),
              make(A.GaussNoise, dict(std_range=(0.03, 0.10)), dict(var_limit=(15.0, 80.0)), p=0.8)]
    elif kind == "backlit":
        ts = [make(A.RandomGamma, dict(gamma_limit=(140, 220)), p=1.0),
              make(A.RandomBrightnessContrast, dict(brightness_limit=(0.0, 0.2), contrast_limit=(-0.5, -0.2)), p=1.0)]
    elif kind == "motion":
        ts = [make(A.MotionBlur, dict(blur_limit=(9, 25)), p=1.0)]
    elif kind == "perspective":
        ts = [make(A.Perspective, dict(scale=(0.04, 0.12)), keep_size=True, p=1.0)]
    else:   # noise and compression
        ts = [make(A.GaussNoise, dict(std_range=(0.05, 0.15)), dict(var_limit=(30.0, 120.0)), p=1.0),
              make(A.ImageCompression, dict(quality_range=(15, 45)), dict(quality_lower=15, quality_upper=45), p=1.0)]
    return A.Compose(ts, bbox_params=bbox), kind

VARIANT_KINDS = ["dark", "backlit", "motion", "perspective", "noise"]
TRANSFORMS = {k: variant_transform(k)[0] for k in VARIANT_KINDS}

if "variants" in RUN_KINDS:
    wr = KindWriter("variants")
    rng = wr.rng
    counts, skipped = Counter(), 0
    tries = 0
    while wr.n < N_VARIANTS and tries < N_VARIANTS * 4:
        tries += 1
        p = rng.choice(with_plates)
        img = load(p)
        boxes = [clip_box(*b) for b in read_boxes(TRAIN_LBL / (p.stem + ".txt"))]
        boxes = [b for b in boxes if b]
        kind = rng.choice(VARIANT_KINDS)
        try:
            r = TRANSFORMS[kind](image=img, bboxes=boxes, labels=[0] * len(boxes))
        except Exception:
            skipped += 1
            continue
        out = r["image"]
        if kind == "backlit":
            out = glare(out, rng)
        if not r["bboxes"]:
            skipped += 1
            continue
        wr.add(out, r["bboxes"], [p.name])
        counts[kind] += 1
    print("by condition:", dict(counts), "| skipped (no plate box left or transform error):", skipped)
    MANIFESTS["variants"] = wr.close(dict(conditions=dict(counts), skipped=skipped))

## 5. `copypaste`: plates pasted onto no-plate images

Plates are cut out of training images (their box, with a small margin) and pasted onto the no-plate training images, mostly the keyboards, at a random size, angle, brightness and position. About one paste in three is pushed to an edge so the plate is **half off the frame**. The box is computed from where the paste landed; a paste with less than 30% of the plate inside the frame is dropped. This teaches the detector that a keyboard is not a plate, and that a cut-off plate is.

In [ ]:
def plate_crops(n, rng):
    crops = []
    for p in rng.sample(with_plates, min(len(with_plates), n * 2)):
        img = load(p)
        h, w = img.shape[:2]
        for cx, cy, bw, bh in read_boxes(TRAIN_LBL / (p.stem + ".txt")):
            mx, my = bw * w * 0.04, bh * h * 0.08
            x0, y0 = int(max(0, cx * w - bw * w / 2 - mx)), int(max(0, cy * h - bh * h / 2 - my))
            x1, y1 = int(min(w, cx * w + bw * w / 2 + mx)), int(min(h, cy * h + bh * h / 2 + my))
            if x1 - x0 >= 24 and y1 - y0 >= 10:
                crops.append((img[y0:y1, x0:x1].copy(), p.name))
        if len(crops) >= n:
            break
    return crops[:n]

def paste_crop(canvas, crop, rng, edge=False, width_frac=None):
    # Warps `crop` onto `canvas` in place. Returns the visible box in pixels (x0, y0, x1, y1), or None.
    H, W = canvas.shape[:2]
    ch, cw = crop.shape[:2]
    tw = (width_frac or rng.choice([rng.uniform(0.05, 0.15), rng.uniform(0.15, 0.55)])) * W
    s = tw / cw
    ang = rng.uniform(-15, 15)
    cx, cy = rng.uniform(0.1, 0.9) * W, rng.uniform(0.1, 0.9) * H
    if edge:
        side = rng.randrange(4)    # centre near an edge: about half the plate is outside
        if side == 0: cx = rng.uniform(-0.1, 0.1) * tw
        elif side == 1: cx = W + rng.uniform(-0.1, 0.1) * tw
        elif side == 2: cy = rng.uniform(-0.1, 0.1) * tw * ch / cw
        else: cy = H + rng.uniform(-0.1, 0.1) * tw * ch / cw
    M = cv2.getRotationMatrix2D((cw / 2, ch / 2), ang, s)
    M[:, 2] += (cx - cw / 2, cy - ch / 2)
    c = np.clip(crop.astype(np.float32) * rng.uniform(0.6, 1.2), 0, 255).astype(np.uint8)
    warped = cv2.warpAffine(c, M, (W, H), flags=cv2.INTER_AREA if s < 1 else cv2.INTER_LINEAR)
    mask = cv2.warpAffine(np.full((ch, cw), 255, np.uint8), M, (W, H))
    corners = cv2.transform(np.float32([[[0, 0], [cw, 0], [cw, ch], [0, ch]]]), M)[0]
    full = abs(cv2.contourArea(corners))
    vis = int((mask > 127).sum())
    ys, xs = np.where(mask > 127)
    if vis == 0 or vis < 0.3 * full or (xs.max() - xs.min()) < 10 or (ys.max() - ys.min()) < 6:
        return None
    m = (cv2.GaussianBlur(mask, (0, 0), 0.8).astype(np.float32) / 255)[..., None]
    canvas[:] = (canvas * (1 - m) + warped * m).astype(np.uint8)
    return float(xs.min()), float(ys.min()), float(xs.max() + 1), float(ys.max() + 1)

if "copypaste" in RUN_KINDS:
    assert no_plates, "no no-plate training images to paste onto"
    wr = KindWriter("copypaste")
    rng = wr.rng
    crops = plate_crops(800, rng)
    print("plate crops:", len(crops))
    edge_n = 0
    while wr.n < N_COPYPASTE:
        bgp = rng.choice(no_plates)
        bg = load(bgp).copy()
        parents, boxes = [bgp.name], []
        for _ in range(1 if rng.random() < 0.8 else 2):
            crop, pname = rng.choice(crops)
            edge = rng.random() < 0.33
            b = paste_crop(bg, crop, rng, edge=edge)
            if b:
                boxes.append(b); parents.append(pname)
                edge_n += int(b[0] <= 0 or b[1] <= 0 or b[2] >= bg.shape[1] or b[3] >= bg.shape[0])
        if boxes:
            wr.add(bg, boxes, parents, norm=False)
    print("pastes touching the frame edge:", edge_n)
    MANIFESTS["copypaste"] = wr.close(dict(edge_touching_boxes=edge_n, plate_crops=len(crops)))

## 6. `screen`: plates on screens

Two ways, half each. **Effect:** a training image with plates gets a moire pattern (two crossed fine gratings), a glare spot and a slight colour shift, as if filmed off a screen. **In a screen:** the whole training photo is shrunk into a tilted screen with a dark bezel on a dim background, with the plate boxes moved by the same transform, then the same effect is applied over the screen. No real screen photos are used.

In [ ]:
def moire(img, rng, strength=None):
    h, w = img.shape[:2]
    yy, xx = np.mgrid[0:h, 0:w].astype(np.float32)
    out = np.zeros((h, w), np.float32)
    for _ in range(2):
        f = rng.uniform(0.25, 0.7)
        t = rng.uniform(0, math.pi)
        out += np.sin(2 * math.pi * f * (xx * math.cos(t) + yy * math.sin(t)))
    out = out / 2 * (strength or rng.uniform(0.04, 0.12))
    return np.clip(img.astype(np.float32) * (1 + out[..., None]), 0, 255).astype(np.uint8)

def colour_shift(img, rng):
    gain = np.array([rng.uniform(0.9, 1.05), rng.uniform(0.92, 1.05), rng.uniform(0.95, 1.12)], np.float32)   # slightly blue
    return np.clip(img.astype(np.float32) * gain + rng.uniform(-6, 10), 0, 255).astype(np.uint8)

def screen_effect(img, rng):
    out = moire(img, rng)
    out = colour_shift(out, rng)
    if rng.random() < 0.7:
        out = glare(out, rng, strength=rng.uniform(70, 150))
    if rng.random() < 0.3:
        out = cv2.GaussianBlur(out, (0, 0), rng.uniform(0.5, 1.2))
    return out

def in_screen(photo, boxes_norm, rng):
    # photo (with plates) shown inside a tilted screen on a dim background. Returns the new image and boxes in pixels.
    H, W = 480, 640
    bg = np.zeros((H, W, 3), np.float32) + np.array([rng.uniform(15, 70)] * 3)
    bg += np.linspace(0, rng.uniform(0, 40), H)[:, None, None]
    bg = np.clip(bg + np.random.default_rng(rng.randrange(1 << 30)).normal(0, 4, bg.shape), 0, 255).astype(np.uint8)
    sw = rng.uniform(0.5, 0.85) * W
    sh = sw * rng.uniform(0.55, 0.75)
    cx, cy = rng.uniform(0.4, 0.6) * W, rng.uniform(0.4, 0.6) * H
    j = 0.05 * sw
    quad = np.float32([[cx - sw / 2, cy - sh / 2], [cx + sw / 2, cy - sh / 2], [cx + sw / 2, cy + sh / 2], [cx - sw / 2, cy + sh / 2]])
    quad += np.float32([[rng.uniform(-j, j), rng.uniform(-j, j)] for _ in range(4)])
    ph, pw = photo.shape[:2]
    src = np.float32([[0, 0], [pw, 0], [pw, ph], [0, ph]])
    Hm = cv2.getPerspectiveTransform(src, quad)
    bez = quad.mean(0) + (quad - quad.mean(0)) * 1.05      # bezel: a slightly larger dark quad
    cv2.fillConvexPoly(bg, bez.astype(np.int32), (12, 12, 14))
    warped = cv2.warpPerspective(photo, Hm, (W, H), flags=cv2.INTER_AREA)
    mask = cv2.warpPerspective(np.full((ph, pw), 255, np.uint8), Hm, (W, H))
    m = (mask.astype(np.float32) / 255)[..., None]
    comp = (bg * (1 - m) + warped * m).astype(np.uint8)
    pts = []
    for cxn, cyn, bw, bh in boxes_norm:
        c = np.float32([[[(cxn - bw / 2) * pw, (cyn - bh / 2) * ph], [(cxn + bw / 2) * pw, (cyn - bh / 2) * ph],
                         [(cxn + bw / 2) * pw, (cyn + bh / 2) * ph], [(cxn - bw / 2) * pw, (cyn + bh / 2) * ph]]])
        q = cv2.perspectiveTransform(c, Hm)[0]
        pts.append((q[:, 0].min(), q[:, 1].min(), q[:, 0].max(), q[:, 1].max()))
    return comp, pts

if "screen" in RUN_KINDS:
    wr = KindWriter("screen")
    rng = wr.rng
    n_eff = n_in = 0
    while wr.n < N_SCREEN:
        p = rng.choice(with_plates)
        photo = load(p)
        boxes = [clip_box(*b) for b in read_boxes(TRAIN_LBL / (p.stem + ".txt"))]
        boxes = [b for b in boxes if b]
        if not boxes:
            continue
        if rng.random() < 0.5:
            wr.add(screen_effect(photo, rng), boxes, [p.name])
            n_eff += 1
        else:
            comp, pts = in_screen(photo, boxes, rng)
            small = [b for b in pts if (b[2] - b[0]) >= 8 and (b[3] - b[1]) >= 5]   # boxes too tiny to learn from are dropped
            if len(small) != len(pts):
                continue
            wr.add(screen_effect(comp, rng), pts, [p.name], norm=False)
            n_in += 1
    print(f"effect on the photo: {n_eff}, photo inside a screen: {n_in}")
    MANIFESTS["screen"] = wr.close(dict(effect_only=n_eff, in_screen=n_in))

## 7. `rendered`: Philippine-style plates with made-up text

Plates are drawn with Pillow: a white plate with a pale band, dark characters, a small "PILIPINAS" or "PHILIPPINES" header and sometimes a region word, in a bold sans font. Car plates are one line (`ABC 1234`, `ABC 123`, `AB 1234`); motorcycle plates are two lines. **Every character is random; no real plate number is used.** Each plate is warped (tilt, perspective), sometimes pushed half off the frame, and placed on a backdrop: half the time a no-plate training image, half the time a procedural scene (gradient, blocks, the rear of a vehicle). It is the generic look, not the LTO design. Keep this kind a small share.

In [ ]:
import string
FONT_FILES = ["/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", "/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf",
              "/usr/share/fonts/truetype/liberation/LiberationSansNarrow-Bold.ttf", "/usr/share/fonts/truetype/dejavu/DejaVuSansCondensed-Bold.ttf"]
FONT_FILES = [f for f in FONT_FILES if Path(f).is_file()]
print("fonts found:", [Path(f).name for f in FONT_FILES] or "none (Pillow default font)")
REGIONS = ["LUZON", "VISAYAS", "MINDANAO", "NCR"]

def get_font(rng, size):
    if FONT_FILES:
        return ImageFont.truetype(rng.choice(FONT_FILES), size)
    return ImageFont.load_default(size)

def plate_text(rng, kind):
    L = lambda n: "".join(rng.choice(string.ascii_uppercase) for _ in range(n))
    D = lambda n: "".join(rng.choice(string.digits) for _ in range(n))
    if kind == "car":
        return [rng.choice([f"{L(3)} {D(4)}", f"{L(3)} {D(3)}", f"{L(2)} {D(4)}"])]
    return [D(3), L(3)] if rng.random() < 0.5 else [L(2), D(5)]

def draw_centered(d, text, f, cx, cy, fill):
    x0, y0, x1, y1 = d.textbbox((0, 0), text, font=f)
    d.text((cx - (x1 - x0) / 2 - x0, cy - (y1 - y0) / 2 - y0), text, font=f, fill=fill)

def render_plate(rng):
    kind = "car" if rng.random() < 0.75 else "moto"
    W, H = (780, 280) if kind == "car" else (470, 270)
    bg = tuple(int(v) for v in np.clip(np.array([245, 246, 242]) + rng.randint(-20, 8), 0, 255))
    ink = rng.choice([(20, 20, 20), (15, 60, 35), (10, 30, 80)])
    im = Image.new("RGB", (W, H), bg)
    d = ImageDraw.Draw(im)
    band = tuple(int(c) for c in rng.choice([(200, 225, 240), (210, 235, 215), (235, 235, 235)]))
    d.rectangle([0, int(H * 0.08), W, int(H * 0.45)], fill=band)
    d.rounded_rectangle([3, 3, W - 4, H - 4], radius=18, outline=ink, width=rng.randint(4, 9))
    small = get_font(rng, int(H * 0.12))
    draw_centered(d, "PILIPINAS" if rng.random() < 0.5 else "PHILIPPINES", small, W / 2, H * 0.13, ink)
    lines = plate_text(rng, kind)
    if len(lines) == 1:
        draw_centered(d, lines[0], get_font(rng, int(H * 0.52)), W / 2, H * 0.55, ink)
    else:
        big = get_font(rng, int(H * 0.34))
        draw_centered(d, lines[0], big, W / 2, H * 0.40, ink)
        draw_centered(d, lines[1], big, W / 2, H * 0.72, ink)
    if rng.random() < 0.7:
        draw_centered(d, rng.choice(REGIONS), small, W / 2, H * 0.90, ink)
    for sx in (0.12, 0.88):
        r = int(H * 0.035)
        d.ellipse([W * sx - r, H * 0.13 - r, W * sx + r, H * 0.13 + r], fill=(150, 150, 150))
    a = np.asarray(im).astype(np.float32)
    if rng.random() < 0.5:     # dirt and wear
        noise = cv2.GaussianBlur(np.random.default_rng(rng.randrange(1 << 30)).random((H, W)).astype(np.float32), (0, 0), 25)
        a *= (1 - 0.25 * (noise - noise.min()) / (np.ptp(noise) + 1e-6))[..., None]
    return np.clip(a, 0, 255).astype(np.uint8)

def rand_color(rng, lo=0, hi=255):
    return tuple(rng.randint(lo, hi) for _ in range(3))

def scene(rng, w, h):
    # procedural backdrop: gradient, blocks, and often the rear of a vehicle. Returns the image and a plate anchor (cx, cy, car width).
    t = np.linspace(0, 1, h, dtype=np.float32)[:, None, None]
    img = np.array(rand_color(rng, 60, 230), np.float32) * (1 - t) + np.array(rand_color(rng, 20, 160), np.float32) * t
    pil = Image.fromarray(np.broadcast_to(img, (h, w, 3)).astype(np.uint8).copy())
    d = ImageDraw.Draw(pil)
    for _ in range(rng.randint(3, 12)):
        x, y = rng.randint(-50, w), rng.randint(-50, h)
        d.rectangle([x, y, x + rng.randint(10, w // 2), y + rng.randint(10, h // 2)], fill=rand_color(rng, 10, 240))
    anchor = None
    if rng.random() < 0.8:
        cw = rng.uniform(0.35, 1.0) * w
        cx, cy = rng.uniform(0.2, 0.8) * w, rng.uniform(0.4, 0.8) * h
        body = rand_color(rng, 10, 240)
        ch = cw * rng.uniform(0.45, 0.7)
        d.rounded_rectangle([cx - cw / 2, cy - ch * 0.7, cx + cw / 2, cy + ch * 0.3], radius=int(cw * 0.06), fill=body)
        d.rectangle([cx - cw / 2, cy + ch * 0.05, cx + cw / 2, cy + ch * 0.3], fill=tuple(int(c * 0.5) for c in body))
        for s in (-1, 1):
            d.rectangle([cx + s * cw * 0.42 - cw * 0.06, cy - ch * 0.35, cx + s * cw * 0.42 + cw * 0.06, cy - ch * 0.2], fill=(170, 20, 20))
        anchor = (cx, cy, cw)
    return np.asarray(pil).copy(), anchor

def paste_rendered(rng, img, anchor):
    # Warps a rendered plate into img. Returns the visible box in pixels, or None.
    h, w = img.shape[:2]
    plate = render_plate(rng)
    ph, pw = plate.shape[:2]
    if anchor and rng.random() < 0.85:
        cx, cy, cw = anchor
        tw = cw * rng.uniform(0.18, 0.3)
    else:
        tw = w * rng.choice([rng.uniform(0.04, 0.12), rng.uniform(0.12, 0.5)])
        cx = rng.uniform(min(tw / 2, w / 2), max(w - tw / 2, w / 2))
        cy = rng.uniform(min(tw / 4, h / 2), max(h - tw / 4, h / 2))
    th = tw * ph / pw
    if rng.random() < 0.2:      # push toward an edge so the plate is cut off
        side = rng.randrange(4)
        if side == 0: cx = rng.uniform(-0.3, 0.1) * tw
        elif side == 1: cx = w - rng.uniform(-0.3, 0.1) * tw
        elif side == 2: cy = rng.uniform(-0.1, 0.1) * th
        else: cy = h - rng.uniform(-0.1, 0.1) * th
    src = np.float32([[0, 0], [pw, 0], [pw, ph], [0, ph]])
    j = 0.12 * tw
    dst = np.float32([[-tw / 2, -th / 2], [tw / 2, -th / 2], [tw / 2, th / 2], [-tw / 2, th / 2]])
    dst += np.float32([[rng.uniform(-j, j), rng.uniform(-j, j) * 0.5] for _ in range(4)])
    ang = math.radians(rng.uniform(-15, 15))
    R = np.float32([[math.cos(ang), -math.sin(ang)], [math.sin(ang), math.cos(ang)]])
    dst = dst @ R.T + np.float32([cx, cy])
    Mx = cv2.getPerspectiveTransform(src, dst)
    warped = cv2.warpPerspective(plate, Mx, (w, h), flags=cv2.INTER_AREA if tw < pw else cv2.INTER_LINEAR)
    mask = cv2.warpPerspective(np.full((ph, pw), 255, np.uint8), Mx, (w, h))
    full = abs(cv2.contourArea(dst))
    ys, xs = np.where(mask > 127)
    if len(xs) == 0 or (mask > 127).sum() < 0.3 * full or xs.max() - xs.min() < 12 or ys.max() - ys.min() < 6:
        return None
    m = (cv2.GaussianBlur(mask, (0, 0), 0.8).astype(np.float32) / 255)[..., None]
    img[:] = (img * (1 - m) + warped * m).astype(np.uint8)
    return float(xs.min()), float(ys.min()), float(xs.max() + 1), float(ys.max() + 1)

if "rendered" in RUN_KINDS:
    wr = KindWriter("rendered")
    rng = wr.rng
    use_real_bg = 0
    while wr.n < N_RENDERED:
        parents = []
        if no_plates and rng.random() < 0.5:
            bgp = rng.choice(no_plates)
            img, anchor = load(bgp).copy(), None
            parents = [bgp.name]
            use_real_bg += 1
        else:
            w, h = rng.choice([(640, 480), (480, 640), (640, 360), (640, 640)])
            img, anchor = scene(rng, w, h)
        boxes = []
        for i in range(1 if rng.random() < 0.85 else 2):
            b = paste_rendered(rng, img, anchor if i == 0 else None)
            if b:
                boxes.append(b)
        if not boxes:
            continue
        if rng.random() < 0.3:      # mild camera look: noise
            img = np.clip(img.astype(np.float32) + np.random.default_rng(rng.randrange(1 << 30)).normal(0, rng.uniform(2, 6), img.shape), 0, 255).astype(np.uint8)
        wr.add(img, boxes, parents, norm=False)
    MANIFESTS["rendered"] = wr.close(dict(on_no_plate_training_images=use_real_bg, plate_text="random, made up"))

## 8. Look at samples

Draws the boxes on a few images from each kind. **Check by eye** that the boxes sit on the plates, that pasted plates are not floating on obviously impossible places, and that no real, unlabeled plate shows in a no-plate backdrop. Delete the folder on Drive and rerun with a new `SUFFIX` if a kind looks wrong.

In [ ]:
import matplotlib.pyplot as plt
def show(kind_folder, n=8, cols=4):
    d = SYN_DIR / kind_folder
    files = sorted((d / "images").glob("*.jpg"))
    pick = random.Random(1).sample(files, min(n, len(files)))
    rows = max(1, (len(pick) + cols - 1) // cols)
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 3.2 * rows))
    for ax in np.ravel(axes):
        ax.axis("off")
    for ax, f in zip(np.ravel(axes), pick):
        im = cv2.cvtColor(cv2.imread(str(f)), cv2.COLOR_BGR2RGB)
        h, w = im.shape[:2]
        for cx, cy, bw, bh in read_boxes(d / "labels" / (f.stem + ".txt")):
            cv2.rectangle(im, (int((cx - bw / 2) * w), int((cy - bh / 2) * h)), (int((cx + bw / 2) * w), int((cy + bh / 2) * h)), (255, 0, 0), max(2, w // 200))
        ax.imshow(im); ax.set_title(f.name, fontsize=7)
    fig.suptitle(kind_folder); plt.tight_layout(); plt.show()

for k in RUN_KINDS:
    show(k + SUFFIX)

## 9. Totals and the leak check

Prints how many synthetic images each folder holds and what share of the training images the folders add up to (protocol: at most about a third to a half; rendered plates a small share). Then `ml.dataset.leak_report(v2, extra_train=[...])` re-checks that no synthetic image is identical or a near duplicate of a validation or test image. **All four counts must be 0.** Synthetic images come only from training images, so a non-zero count means the dataset itself has a problem; do not train until it is 0.

In [ ]:
folders = [SYN_DIR / (k + SUFFIX) for k in RUN_KINDS]
n_train = len(train_files)
tot = 0
print("| folder | images | boxes | no-plate images | share of v2 training images |")
print("|---|---|---|---|---|")
for d in folders:
    m = json.loads((d / "manifest.json").read_text())
    tot += m["images"]
    print(f"| `{d.name}` | {m['images']} | {m['boxes']} | {m['no_plate_images']} | {m['images'] / n_train:.1%} |")
print(f"\nall kinds together: {tot} synthetic images next to {n_train} training images = {tot / (n_train + tot):.1%} of the training data")
leaks = ds.leak_report(V2, extra_train=folders)
print(json.dumps(leaks, indent=2))
assert all(v == 0 for v in leaks.values()), "leak found: do not train on these folders"
print("leak check passed")